# Bronze Layer

## Data Optimization

Purpose:

- Optimize Bronze datasets, per entity
- Improve read performance
- Reduce the number of small files
- Prepare datasets for downstream processing

## Environment Bootstrap

Required on Databricks Free Edition: there is no `libraries: - whl: ...` mechanism wired to a classic/serverless cluster here, so the project wheel has to be pip-installed explicitly in the notebook before any `data_platform`/`integrations` import works.

`wheel_path` comes from a Job base_parameter (`${workspace.root_path}/artifacts/.internal`, resolved by the Databricks bundle at deploy time) -- portable across users and targets (dev/prod), unlike a hardcoded `/Workspace/Users/<you>/...` path. Bundle substitutions only expand inside bundle YAML files, never inside notebook content directly, which is why this goes through a widget instead of being inlined here.

In [ ]:
dbutils.widgets.text("wheel_path", "")
wheel_path = dbutils.widgets.get("wheel_path")
wheel_glob = f"{wheel_path}/*.whl"

%pip install $wheel_glob

dbutils.library.restartPython()

## Imports

In [ ]:
from data_platform.compute.delta_io import read_delta, write_delta
from data_platform.compute.spark import get_spark
from data_platform.storage.config import StorageConfig
from integrations.databricks.runtime.parameters import get_parameter

## Parameters

`entities` is a comma-separated list (e.g. `customers,orders,products`), parsed here -- a single value still works (`"customers".split(",")` == `["customers"]`). Processing multiple entities in one notebook execution means the wheel install and SparkSession above are paid once per run, not once per entity.

In [ ]:
entities = [
    entity.strip()
    for entity in get_parameter("entities", default="customers").split(",")
    if entity.strip()
]

## Spark Session

Created once, reused across every entity in the loop below.

In [ ]:
spark = get_spark("Bronze Optimization")

## Optimize Function

In [ ]:
def optimize_entity(spark, entity: str) -> None:
    bronze_df = read_delta(spark, StorageConfig.bronze(entity))

    bronze_df.printSchema()
    bronze_df.show(10)

    file_count = spark.sql(
        f"DESCRIBE DETAIL delta.`{StorageConfig.bronze(entity)}`"
    ).first()["numFiles"]

    print(f"Files: {file_count}")

    optimized_df = bronze_df.coalesce(1)

    write_delta(optimized_df, StorageConfig.bronze(entity), mode="overwrite")

    optimized_validation_df = read_delta(spark, StorageConfig.bronze(entity))

    print(f"Total records: {optimized_validation_df.count()}")

    optimized_file_count = spark.sql(
        f"DESCRIBE DETAIL delta.`{StorageConfig.bronze(entity)}`"
    ).first()["numFiles"]

    print(f"Files: {optimized_file_count}")

## Run For Each Entity

Entities are independent of each other, so one failing must not cost reprocessing the others: each is wrapped in its own try/except, errors are collected, and only surface as a single aggregated failure at the end -- which still fails the Databricks task for real (no silent partial success).

In [ ]:
errors: dict[str, str] = {}

for entity in entities:
    print(f"=== {entity}: optimizing ===")

    try:
        optimize_entity(spark, entity)

    except Exception as error:
        print(f"[{entity}] FAILED: {error}")
        errors[entity] = str(error)

if errors:
    summary = "\n".join(
        f"  - {entity}: {message}" for entity, message in errors.items()
    )

    raise RuntimeError(
        f"{len(errors)} of {len(entities)} entities failed:\n{summary}"
    )

print(f"OK: {len(entities)} entities processed: {', '.join(entities)}")